# 00 - Setup do ambiente no Google Colab

Este notebook prepara o runtime do Colab para executar os experimentos do projeto:
1. Verifica a GPU (Tesla T4 ou superior),
2. Clona o repositório,
3. Instala as dependências (`requirements-colab.txt`),
4. Carrega o dataset ToLD-BR via Hugging Face (sem Google Drive),
5. Fornece uma célula para baixar resultados ao final da sessão,

## 1. Verificar GPU

In [1]:
!nvidia-smi

Sun Nov  1 19:24:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
    print("Versão do PyTorch: ", torch.__version__)
    print("CUDA: ", torch.version.cuda)

CUDA disponível: True
GPU:  Tesla T4
Versão do PyTorch:  2.11.0+cu130
CUDA:  13.0


## 2. Clonar o Repósitório

In [3]:
import os

REPO_URL = "https://github.com/elianisaac/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais.git"
REPO_DIR = "Classifica-o-de-Discurso-T-xico-em-Redes-Sociais"

# Só clona se a pasta NÃO existir
if not os.path.exists(f"/content/{REPO_DIR}"):
    !git clone {REPO_URL} /content/{REPO_DIR}

# Força a entrada no caminho absoluto correto
%cd /content/{REPO_DIR}

Cloning into '/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 3), reused 17 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 143.82 KiB | 5.53 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais


## 3. Instalar Dependencias

Usa `requirements-colab.txt`


In [4]:
# Atualiza as ferramentas de build do Python
!pip install --upgrade pip setuptools wheel

# Instala as dependencias
%pip install -q -r requirements-collab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 8.2 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.2.1 which is incompatible.


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.


In [5]:
# Verificação rápida das versões instaladas
import datasets, transformers, sklearn
print("datasets:", datasets.__version__)
print("transformers: ", transformers.__version__)
print("scikit-learn: ", sklearn.__version__)

datasets: 2.21.0
transformers:  5.21.0
scikit-learn:  1.6.1


# 4. Carregamento do Dataset

O script `src/load_data.py` baixa o dataset do Hugging Face (`JAugusto97/told-br`) e salva em disco (`data/processed/`) para evitar re-executar o *loading script* a cada uso.

In [6]:
!python src/load_data.py --variant binary --output data/processed/

[2026-11-01 19:25:12] [INFO] [load_data] ============================================================
[2026-11-01 19:25:12] [INFO] [load_data] Processando variante: binary
[2026-11-01 19:25:12] [INFO] [load_data] ============================================================
[2026-11-01 19:25:12] [INFO] [load_data] Baixando do Hugging Face: JAugusto97/told-br (config=binary)
Generating train split: 100% 16800/16800 [00:00<00:00, 50307.68 examples/s]
Generating test split: 100% 2100/2100 [00:00<00:00, 50356.97 examples/s]
Generating validation split: 100% 2100/2100 [00:00<00:00, 52307.37 examples/s]
[2026-11-01 19:25:15] [INFO] [load_data] Salvando em disco: data/processed/told-br-binary
Saving the dataset (1/1 shards): 100% 16800/16800 [00:00<00:00, 1448035.58 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 543203.11 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 579552.47 examples/s]
[2026-11-01 19:25:15] [INFO] [load_data] Resumo do 

In [7]:
# Inspeção rápida do dataset carregado
from datasets import load_from_disk
ds = load_from_disk("data/processed/told-br-binary")
print(ds)
print("Exemplo: ", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16800
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2100
    })
})
Exemplo:  {'text': 'rt @user olha quem chegouuuuu, nossos queridinhos!!! vem na minha direção 🤩\n\nfem 👧: 10/15\nmasc 👦: 25/40\n\n#qqmt https://t.co/jivoqeuq…', 'label': 0}


In [8]:
# Roda o pré-processamento gerando os textos limpos
!python src/preprocessing.py \
    --input data/processed/told-br-binary \
    --output data/processed/told-br-binary-clean

[2026-11-01 19:25:20] [INFO] [preprocessing] Carregando dataset de: data/processed/told-br-binary
[2026-11-01 19:25:20] [INFO] [preprocessing] Aplicando pré-processamento nas partições...
[2026-11-01 19:25:20] [INFO] [preprocessing]   Processando split: train
Map (num_proc=2): 100% 16800/16800 [00:01<00:00, 12962.23 examples/s]
[2026-11-01 19:25:21] [INFO] [preprocessing]   Processando split: test
Map (num_proc=2): 100% 2100/2100 [00:00<00:00, 8504.11 examples/s]
[2026-11-01 19:25:21] [INFO] [preprocessing]   Processando split: validation
Map (num_proc=2): 100% 2100/2100 [00:00<00:00, 8022.59 examples/s]
[2026-11-01 19:25:22] [INFO] [preprocessing] Salvando dataset processado em: data/processed/told-br-binary-clean
Saving the dataset (1/1 shards): 100% 16800/16800 [00:00<00:00, 1189411.53 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 445976.63 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 458560.93 examples/s]
[2026-11-01 19:25:22

## 5. Executar os experimentos

Com o ambiente pronto e o dataset carregado, rode os scripts de treino. Exemplos:"

In [9]:
# Baseline: Regressão Logística
!python src/train_logistic.py --data data/processed/ --output results

python3: can't open file '/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/src/train_logistic.py': [Errno 2] No such file or directory


In [10]:
# BERTimbau (fine-tuning)
!python src/train_bertimbau.py
--data data/processed/
--model neuralmind/bert-base-portuguese-cased
--epochs 5
--batch-size 16
--output-dir results/models/

SyntaxError: invalid syntax (1041307888.py, line 3)

In [ ]:
# Avaliação e comparação
!python src/evaluate.py --results results/ --output results/metrics/

## 6. Baixar resultados ao final da sessão

In [ ]:
import shutil
from google.colab import files

# Compacta a pasta de resultados
archive_path = shutil.make_archive("/content/results_backup", "zip", "/content/[nome-do-repo]/results")
print("Arquivo criado:", archive_path)

# Baixa o arquivo para o computador local
files.download(archive_path)